# Customer Segmentation with KMeans
### Original example by Danny Barren: *ML for Business Insights, Examples*

**The business question:** *What natural groups of customers do we have, and how should we treat each group differently?*

There is no "right answer" column here. This is **unsupervised** learning: KMeans groups customers who behave alike, and **we** give each group a name and an action. The notebook walks through:

1. Preparing behaviour data (fill gaps, tame skewed columns, put everything on the same scale)
2. Choosing the number of segments with the **elbow** method and the **silhouette** score
3. Checking the segments are **stable** (not an accident of one random start)
4. **Profiling** each segment in plain language, against the average customer
5. An **action table** with what to do with each segment, plus a simple rulebook to assign new customers

**How to use it:** Edit only the **CONFIG** cell, then run top to bottom.
If `CSV_PATH = None`, a **synthetic** customer base is generated for the demo.

In [ ]:
# ============================================================
# IMPORTS (scikit-learn 1.2+)
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.tree import DecisionTreeClassifier, export_text

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Libraries loaded.")

In [ ]:
# ============================================================
# CONFIG: THE ONLY CELL YOU NEED TO EDIT
# ============================================================
CSV_PATH = None                     # e.g. "/content/customers.csv"; None = synthetic demo

ID_COLUMN = "customer_id"

# Behaviour columns used to FORM the segments (numbers only)
CLUSTER_COLUMNS = ["annual_spend", "orders_12m", "avg_order_value", "days_since_last_order",
                   "pct_orders_discounted", "support_contacts_12m", "tenure_months"]
# Skewed money/count columns: a log transform stops a few huge customers from dominating
LOG_COLUMNS = ["annual_spend", "orders_12m", "avg_order_value", "support_contacts_12m"]
# Extra columns only used to DESCRIBE segments afterwards (not to form them)
PROFILE_ONLY_COLUMNS = ["preferred_channel", "region"]
VALUE_COLUMN = "annual_spend"       # used to show each segment's share of revenue

K_RANGE = range(2, 9)               # segment counts to try
MIN_K_FOR_BUSINESS = 3              # fewer than 3 segments is rarely actionable
FORCE_K = None                      # set e.g. 4 to override the automatic choice

DATA_SIZE_MODE = "auto"             # "auto", "small", "large" (large -> MiniBatchKMeans + sampled silhouette)
SILHOUETTE_SAMPLE = 3_000           # rows sampled for the silhouette score on large data
RANDOM_STATE = 42

In [ ]:
# ============================================================
# LOAD DATA (or build a synthetic customer base)
# ============================================================
def make_demo_customers(n=600, seed=RANDOM_STATE):
    """Synthetic customers drawn from four hidden personas:
    VIP loyalists, bargain hunters, new/occasional buyers, and lapsing customers.
    (KMeans never sees these labels; we check later whether it rediscovers them.)"""
    rng = np.random.default_rng(seed)
    persona = rng.choice(["vip", "bargain", "new", "lapsing"], n, p=[0.15, 0.30, 0.30, 0.25])
    p = lambda **kw: np.select([persona == k for k in kw], list(kw.values()))
    orders = np.maximum(1, rng.poisson(p(vip=24, bargain=10, new=3, lapsing=5)))
    aov = np.round(rng.lognormal(np.log(p(vip=140, bargain=45, new=70, lapsing=80)), 0.3), 2)
    df = pd.DataFrame({
        "customer_id": np.arange(20_001, 20_001 + n),
        "orders_12m": orders,
        "avg_order_value": aov,
        "annual_spend": np.round(orders * aov, 2),
        "days_since_last_order": np.clip(rng.normal(p(vip=12, bargain=30, new=25, lapsing=170), p(vip=8, bargain=15, new=15, lapsing=50)), 0, 365).round(),
        "pct_orders_discounted": np.clip(rng.normal(p(vip=0.10, bargain=0.70, new=0.30, lapsing=0.25), 0.10), 0, 1).round(2),
        "support_contacts_12m": rng.poisson(p(vip=2, bargain=1, new=1.5, lapsing=3)),
        "tenure_months": np.clip(rng.normal(p(vip=48, bargain=30, new=4, lapsing=36), p(vip=15, bargain=12, new=2, lapsing=14)), 1, 120).round(),
        "preferred_channel": np.where(rng.random(n) < p(vip=0.3, bargain=0.8, new=0.6, lapsing=0.5), "online", "store"),
        "region": rng.choice(["north", "south", "east", "west"], n),
    })
    df.loc[rng.random(n) < 0.03, "support_contacts_12m"] = np.nan
    df["_hidden_persona"] = persona          # for the demo check only; never used for clustering
    return df

if CSV_PATH is None:
    df = make_demo_customers()
    print("Using SYNTHETIC demo data (CSV_PATH is None).")
else:
    df = pd.read_csv(CSV_PATH)
    print(f"Loaded {CSV_PATH}")
print(f"{len(df):,} customers x {df.shape[1]} columns")
df.head()

In [ ]:
# ============================================================
# CONFIG CHECK + LIGHT EDA
# ============================================================
missing = [c for c in CLUSTER_COLUMNS + PROFILE_ONLY_COLUMNS if c not in df.columns]
assert not missing, f"These CONFIG columns are not in your file: {missing}"
assert set(LOG_COLUMNS) <= set(CLUSTER_COLUMNS), "LOG_COLUMNS must also be in CLUSTER_COLUMNS"

X = df[CLUSTER_COLUMNS].apply(pd.to_numeric, errors="coerce").astype(float)
neg = [c for c in LOG_COLUMNS if (X[c] < 0).any()]
assert not neg, f"Log columns can't contain negative values: {neg}"

display(pd.DataFrame({"missing_%": X.isna().mean() * 100, "mean": X.mean(), "median": X.median(),
                      "max": X.max()}).round(2))
X.hist(bins=30, figsize=(14, 6), layout=(2, 4), color="slateblue"); plt.suptitle("Behaviour columns (long right tails → log)")
plt.tight_layout(); plt.show()

## 1. Prepare the data
KMeans groups customers by **distance**, so every column must be on the same scale; otherwise "annual spend in dollars" would drown out "share of orders discounted".
- fill gaps with the median
- `log1p` for skewed money/count columns
- standardize everything (mean 0, typical step 1)

Bundling these steps in a **Pipeline** means new customers are prepared exactly the same way later.

In [ ]:
other_cols = [c for c in CLUSTER_COLUMNS if c not in LOG_COLUMNS]
prep = ColumnTransformer([
    ("log", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                      ("scale", StandardScaler())]), LOG_COLUMNS),
    ("plain", Pipeline([("impute", SimpleImputer(strategy="median")),
                        ("scale", StandardScaler())]), other_cols),
], verbose_feature_names_out=False)
Z = prep.fit_transform(X)

size_mode = DATA_SIZE_MODE if DATA_SIZE_MODE != "auto" else ("small" if len(df) < 5_000 else "large")
def make_kmeans(k, seed=RANDOM_STATE):
    if size_mode == "large":
        return MiniBatchKMeans(n_clusters=k, n_init=5, batch_size=2_048, random_state=seed)
    return KMeans(n_clusters=k, n_init=20, random_state=seed)
sil_sample = SILHOUETTE_SAMPLE if size_mode == "large" and len(Z) > SILHOUETTE_SAMPLE else None
print(f"Prepared matrix: {Z.shape} | mode: {size_mode} ({type(make_kmeans(2)).__name__})")

## 2. How many segments? Elbow + silhouette
- **Elbow (inertia):** how tightly customers sit around their segment centre. It always improves with more segments; we look for the point where the improvement **slows down**.
- **Silhouette (−1 to 1):** how clearly each customer belongs to its own segment rather than the next one. Higher is better; above ~0.25 is usable for behavioural data.

**Rule used here:** the highest silhouette among K ≥ `MIN_K_FOR_BUSINESS`, with the elbow as a second opinion. You can always override with `FORCE_K`.

In [ ]:
rows = []
for k in K_RANGE:
    km = make_kmeans(k).fit(Z)
    rows.append({"k": k, "inertia": km.inertia_,
                 "silhouette": silhouette_score(Z, km.labels_, sample_size=sil_sample, random_state=RANDOM_STATE)})
k_tbl = pd.DataFrame(rows)

# Elbow point ("kneedle" idea): the K farthest below the straight line from first to last point
xs = (k_tbl["k"] - k_tbl["k"].min()) / (k_tbl["k"].max() - k_tbl["k"].min())
ys = (k_tbl["inertia"] - k_tbl["inertia"].min()) / (k_tbl["inertia"].max() - k_tbl["inertia"].min())
elbow_k = int(k_tbl.loc[((1 - xs) - ys).idxmax(), "k"])
eligible = k_tbl[k_tbl["k"] >= MIN_K_FOR_BUSINESS]
sil_k = int(eligible.loc[eligible["silhouette"].idxmax(), "k"])
K = FORCE_K or sil_k

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(k_tbl["k"], k_tbl["inertia"], "o-"); axes[0].axvline(elbow_k, ls="--", c="grey")
axes[0].set_title(f"Elbow (suggests K={elbow_k})"); axes[0].set_xlabel("K")
axes[1].plot(k_tbl["k"], k_tbl["silhouette"], "o-", c="darkorange"); axes[1].axvline(K, ls="--", c="grey")
axes[1].set_title(f"Silhouette (chosen K={K})"); axes[1].set_xlabel("K")
plt.tight_layout(); plt.show()
display(k_tbl.round(3))
print(f"Elbow suggests {elbow_k}, silhouette suggests {sil_k} -> using K = {K}"
      + (" (forced in CONFIG)" if FORCE_K else ""))

## 3. Fit the final segmentation and check it's stable
A trustworthy segmentation shouldn't change much if we restart KMeans with different random seeds. We compare each re-run with the main result using the **Adjusted Rand Index** (1.0 = identical groupings, 0 = unrelated).

In [ ]:
seg_model = Pipeline([("prep", prep), ("kmeans", make_kmeans(K))]).fit(X)
labels = seg_model.named_steps["kmeans"].labels_

aris = [adjusted_rand_score(labels, make_kmeans(K, seed=s).fit(Z).labels_) for s in range(1, 6)]
# Also refit on random 80% subsamples and compare on the shared customers
rng = np.random.default_rng(RANDOM_STATE)
for _ in range(5):
    idx = rng.choice(len(Z), int(0.8 * len(Z)), replace=False)
    aris.append(adjusted_rand_score(labels[idx], make_kmeans(K).fit(Z[idx]).labels_))
print(f"Stability (ARI across 5 restarts + 5 subsamples): mean {np.mean(aris):.2f}, min {np.min(aris):.2f}")
print("Stable: safe to build actions on." if np.min(aris) > 0.8 else
      "Somewhat unstable: try a different K, fewer columns, or more data before acting on fine details.")
if "_hidden_persona" in df.columns:
    print(f"(Demo only) Match with the hidden personas, ARI: {adjusted_rand_score(df['_hidden_persona'], labels):.2f}")

## 4. Segment profiles: who is in each group?
We compare each segment's average with the overall average as an **index** (100 = average, 150 = 50% above average, 50 = half the average). Indexes make very different columns easy to read side by side.

In [ ]:
prof = X.assign(segment=labels).groupby("segment").mean()
overall = X.mean()
index = (prof / overall * 100).round(0)
size = pd.Series(labels).value_counts().sort_index()
summary = prof.copy()
summary.insert(0, "customers", size)
summary.insert(1, "share_%", (size / size.sum() * 100).round(1))
if VALUE_COLUMN in X.columns:
    rev = X[VALUE_COLUMN].groupby(labels).sum()
    summary.insert(2, "revenue_share_%", (rev / rev.sum() * 100).round(1))
display(summary)

plt.figure(figsize=(10, 0.6 * K + 2))
sns.heatmap(index, annot=True, fmt=".0f", cmap="RdBu_r", center=100, cbar_kws={"label": "index vs average"})
plt.title("Segment profile index (100 = average customer)"); plt.tight_layout(); plt.show()

for col in PROFILE_ONLY_COLUMNS:       # describe segments with columns NOT used to form them
    display(pd.crosstab(labels, df[col], normalize="index").round(2).rename_axis(index="segment", columns=col))

In [ ]:
# Plain-language descriptions: the 2 most "above average" and 2 most "below average" traits per segment
def describe(seg):
    row = index.loc[seg]
    hi = [f"high {c.replace('_', ' ')} ({row[c]:.0f})" for c in row.sort_values(ascending=False).index[:2] if row[c] > 115]
    lo = [f"low {c.replace('_', ' ')} ({row[c]:.0f})" for c in row.sort_values().index[:2] if row[c] < 85]
    return ", ".join(hi + lo) or "close to average on everything"

descriptions = {s: describe(s) for s in index.index}
for s, d in descriptions.items():
    print(f"Segment {s} ({size[s]} customers, {size[s] / size.sum():.0%}): {d}")

## 5. Action table: name each segment and decide what to do
The rules below turn the profile index into a suggested **name** and **action**. They use the demo column names, so edit them to match your own columns and strategy. The first rule that matches wins.

In [ ]:
# (column, condition on index, suggested name, suggested action)
ACTION_RULES = [
    ("days_since_last_order", lambda v: v > 150, "Lapsing", "Win-back offer within 2 weeks; ask why they left"),
    ("annual_spend",          lambda v: v > 150, "VIP / high value", "Loyalty perks, early access, account manager; avoid blanket discounts"),
    ("pct_orders_discounted", lambda v: v > 140, "Bargain hunters", "Targeted promos only; test reducing discount depth to protect margin"),
    ("tenure_months",         lambda v: v < 50,  "New customers", "Onboarding series and second-purchase incentive"),
]
actions = []
for s in index.index:
    name, action = "Core / steady", "Maintain; cross-sell related products"
    for col, cond, n, a in ACTION_RULES:
        if col in index.columns and cond(index.loc[s, col]):
            name, action = n, a
            break
    actions.append({"segment": s, "suggested_name": name, "customers": size[s],
                    "revenue_share_%": summary.loc[s].get("revenue_share_%", np.nan),
                    "profile": descriptions[s], "suggested_action": action})
action_table = pd.DataFrame(actions).set_index("segment")
with pd.option_context("display.max_colwidth", 80):
    display(action_table)

In [ ]:
# A 2-D picture of the segments (PCA squeezes all columns into two axes; some overlap is normal)
pts = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(Z)
plt.figure(figsize=(6.5, 5))
for s in index.index:
    m = labels == s
    plt.scatter(pts[m, 0], pts[m, 1], s=12, alpha=0.6, label=f"{s}: {action_table.loc[s, 'suggested_name']}")
plt.legend(fontsize=8); plt.title("Customer segments (PCA map)"); plt.xlabel("PC1"); plt.ylabel("PC2"); plt.show()

In [ ]:
# Simple rulebook: a shallow decision tree that mimics the segments using the ORIGINAL columns,
# so staff can assign a new customer without running any code.
tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=RANDOM_STATE)
X_filled = X.fillna(X.median())
tree.fit(X_filled, labels)
print(f"The rulebook agrees with KMeans on {tree.score(X_filled, labels):.0%} of customers.\n")
seg_names = [f"segment {c}: {action_table.loc[c, 'suggested_name']}" for c in tree.classes_]
print(export_text(tree, feature_names=CLUSTER_COLUMNS, class_names=seg_names, decimals=1))

## 6. What to do next: business recommendations
- **Give each segment an owner and one action** from the table above; keep messages and offers different per segment.
- **Protect the high-value segment first:** it usually holds a big share of revenue in a small share of customers.
- **Win back lapsing customers early;** the longer they're gone, the harder they are to recover.
- **Test before rolling out:** for each segment, hold back a random 10–20% as a control group and compare spend after 6–8 weeks.
- **Refresh monthly or quarterly:** re-score customers with the saved pipeline and track movement between segments (e.g. *New → Core* is good, *VIP → Lapsing* is an alarm).

**Small vs. large data**
- *Small (hundreds of customers):* keep K small (3–5), use many restarts (`n_init=20`), and check stability before acting.
- *Large (100k+ customers):* `large` mode switches to `MiniBatchKMeans` and a sampled silhouette; fit on a sample and assign everyone else with `seg_model.predict()`.

In [ ]:
# Optional: export segment labels and save the pipeline for scoring new customers
out = df[[ID_COLUMN]].copy() if ID_COLUMN in df.columns else pd.DataFrame(index=df.index)
out["segment"] = labels
out["segment_name"] = out["segment"].map(action_table["suggested_name"])
out["suggested_action"] = out["segment"].map(action_table["suggested_action"])
out.to_csv("customer_segments.csv", index=False)
import joblib
joblib.dump({"model": seg_model, "columns": CLUSTER_COLUMNS, "names": action_table["suggested_name"].to_dict()},
            "segmentation_model.joblib")
print("Saved customer_segments.csv and segmentation_model.joblib")
out.head()